In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

from catboost import CatBoostRegressor

In [3]:
data = pd.read_parquet("../data/processed/move_ru_clean.parquet")
data.head() 

,url,listing_id,title,deal_type,object_type,price,rooms,area_total,area_kitchen,area_living,...,seller_type,photo_count,image_urls,is_studio,has_balcony,has_elevator,has_parking,has_renovation,description,parse_errors
0,https://move.ru/objects/moskva_pereulok_simono...,9293574125,"3-комнатной квартиры, 87.3 м²",sale,apartment,55000000.0,3.0,87.3,13.3,49.3,...,owner,58.0,['https://static-i6.move.ru/images/items/10002...,0.0,0.0,0.0,0.0,1.0,вп - 2694. легкая альтернатива. 1 взрослый соб...,[]
1,https://move.ru/objects/moskva_proezd_kutuzovs...,9287887608,"2-комнатная квартира, 63.8 м²",sale,apartment,67500185.0,2.0,63.8,20.7,25.0,...,NaN,14.0,['https://static-i6.move.ru/images/items/10009...,0.0,0.0,1.0,1.0,0.0,премиальная жизнь на берегу москвы-реки почему...,[]
2,https://move.ru/objects/moskva_prospekt_40_let...,9287820819,"1-комнатную квартиру, 27.1 м²",sale,apartment,16500000.0,1.0,27.1,8.0,12.0,...,owner,38.0,['https://static-i6.move.ru/images/items/10009...,0.0,0.0,1.0,1.0,1.0,собственник. дом клубного плана. продам уютную...,[]
3,https://move.ru/objects/moskva_prospekt_60-let...,9293539455,"1-комнатной квартиры, 34 м²",sale,apartment,21200000.0,1.0,34.0,8.0,24.0,...,owner,46.0,['https://static-i6.move.ru/images/items/10002...,0.0,1.0,0.0,0.0,1.0,вп - 2695. свободная продажа. 1 взрослый собст...,[]
4,https://move.ru/objects/moskva_prospekt_kutuzo...,9284926509,"3-комнатную квартиру, 112 м²",sale,apartment,142000000.0,3.0,112.0,32.0,80.0,...,developer,28.0,['https://static-i6.move.ru/images/items/10009...,0.0,0.0,0.0,1.0,NaN,жилой комплекс бадаевский 112 м2 клубный дом. ...,[]


In [4]:
data.shape

(14534, 31)

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 14534 entries, 0 to 14533
Data columns (total 31 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   url              14534 non-null  str    
 1   listing_id       14534 non-null  str    
 2   title            14534 non-null  str    
 3   deal_type        14534 non-null  str    
 4   object_type      14534 non-null  str    
 5   price            14534 non-null  float64
 6   rooms            14534 non-null  float64
 7   area_total       14534 non-null  float64
 8   area_kitchen     12563 non-null  float64
 9   area_living      14229 non-null  float64
 10  price_per_m2     14534 non-null  float64
 11  address          12958 non-null  str    
 12  okrug            14298 non-null  str    
 13  district         14534 non-null  str    
 14  metro            14534 non-null  str    
 15  metro_time_min   14533 non-null  float64
 16  floor            14534 non-null  float64
 17  total_floors     14534 

In [6]:
data.isna().sum()

url                    0
listing_id             0
title                  0
deal_type              0
object_type            0
price                  0
rooms                  0
area_total             0
area_kitchen        1971
area_living          305
price_per_m2           0
address             1576
okrug                236
district               0
metro                  0
metro_time_min         1
floor                  0
total_floors           0
house_type         14169
year_built         12911
completion_year     4525
seller_type         1614
photo_count            0
image_urls             0
is_studio              0
has_balcony            0
has_elevator           0
has_parking            0
has_renovation      8977
description            0
parse_errors           0
dtype: int64

In [7]:
target = "price_per_m2"

drop_cols = [
    "price",          # leakage через area_total
    "price_per_m2",   # target
    "url",
    "listing_id",
    "image_urls",
    "parse_errors",
    "title",
    "description",    
]

num_features = [
    "rooms",
    "area_total",
    "area_kitchen",
    "area_living",
    "metro_time_min",
    "floor",
    "total_floors",
    "year_built",
    "completion_year",
    "photo_count",
    "is_studio",
    "has_balcony",
    "has_elevator",
    "has_parking",
    "has_renovation",
]

cat_features = [
    "okrug",
    "district",
    "metro",
    "house_type",
    "seller_type",
]

In [8]:
data[num_features] = data[num_features].fillna(-1)
data[cat_features] = data[cat_features].fillna("unknown")

In [9]:
X = data[num_features + cat_features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
lr = LinearRegression()
lr.fit(X_train[num_features], y_train)

preds = lr.predict(X_test[num_features])

mae = mean_absolute_error(y_test, preds)
mape = mean_absolute_percentage_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", mae)
print("MAPE:", mape)
print("R^2", r2)

MAE: 149777.52910720816
MAPE: 0.35794027866704786
R^2 0.3488037023821281


In [11]:
cat_features_index = [X.columns.get_loc(col) for col in cat_features]

model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features_index
)

0:	learn: 258380.8444499	total: 64.4ms	remaining: 19.3s
50:	learn: 100703.8341615	total: 212ms	remaining: 1.04s
100:	learn: 85472.9400934	total: 362ms	remaining: 713ms
150:	learn: 76821.9993398	total: 541ms	remaining: 534ms
200:	learn: 70565.5859189	total: 748ms	remaining: 369ms
250:	learn: 66059.1646308	total: 1.05s	remaining: 205ms
299:	learn: 62843.3163284	total: 1.21s	remaining: 0us


CatBoostRegressor(depth=6, iterations=300, learning_rate=0.1, loss_function='RMSE', verbose=50)

In [12]:
preds_cat = model.predict(X_test)

In [13]:
from sklearn.metrics import r2_score


mae = mean_absolute_error(y_test, preds_cat)
mape = mean_absolute_percentage_error(y_test, preds_cat)
r2 = r2_score(y_test, preds_cat)

print("MAE:", mae)
print("MAPE:", mape)
print("R^2", r2)

MAE: 39676.210750170314
MAPE: 0.09020954215363476
R^2 0.9388359895263971
